# Pydantic

**Pydantic** is a Python library for data validation and settings management using type annotations. It validates/parses data into Python objects and raises clear errors on bad input.
Use cases:
- Validating API request/response payloads
- Parsing/cleaning config & env variables
- Deserializing JSON into typed models
- Data pipelines where inputs are untrusted
Example:
    from pydantic import BaseModel, EmailStr, field_validator

    class User(BaseModel):
        name: str
        age: int
        email: EmailStr

        @field_validator("age")
        def age_must_be_positive(cls, v):
            if v < 0:
                raise ValueError("age must be >= 0")
            return v

    ### Valid
    u = User(name="Alice", age=30, email="a@b.com")
    print(u.model_dump())

    ### Invalid -> ValidationError
    User(name="Bob", age=-5, email="not-an-email")
    Config / env example:
    from pydantic import BaseModel

    class Settings(BaseModel):
        db_url: str
        debug: bool = False

    s = Settings(db_url="sqlite:///db.sqlite")  # env vars also auto-loaded


Pydantic v2 (current) uses model_validate, model_dump(), and pydantic-settings for env config.

In [1]:
from pydantic import BaseModel

In [3]:
from dataclasses import dataclass

@dataclass
class Person():
    name:str
    age:int
    city:str

person=Person(name="Mew",age=35,city="Alaska")
print(person)


Person(name='Mew', age=35, city='Alaska')


In [4]:
person=Person(name="Krish",age=35,city=35)
print(person)

Person(name='Krish', age=35, city=35)


In [5]:
class Person1(BaseModel):
    name:str
    age:int
    city:str

person=Person1(name="Krish",age=35,city="Bangalore")
print(person)




name='Krish' age=35 city='Bangalore'


In [6]:
person1=Person1(name="Krish",age=35,city=12)
print(person1)

ValidationError: 1 validation error for Person1
city
  Input should be a valid string [type=string_type, input_value=12, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type

#### 2. Model with Optional Fields
Add optional fields using Python's Optional type:



In [7]:
from typing import Optional
class Employee(BaseModel):
    id: int
    name: str
    department: str
    salary: Optional[float] = None  # Optional with default value
    is_active: Optional[bool] = True  # Optional with default True


In [8]:
# Examples with and without optional fields
emp1 = Employee(id=1, name="John", department="IT")
print(emp1)  # id=1 name='John' department='IT' salary=None is_active=True

id=1 name='John' department='IT' salary=None is_active=True


In [9]:
emp2 = Employee(id=2, name="Jane", department="HR", salary=60000, is_active=False)
print(emp2)

id=2 name='Jane' department='HR' salary=60000.0 is_active=False


Definition:
- Optional[type]: Indicates the field can be None

- Default value (= None or = True): Makes the field optional

- Required fields must still be provided

- Pydantic validates types even for optional fields when values are provided



In [10]:
from pydantic import BaseModel
from typing import List

class Classroom(BaseModel):
    room_number: str
    students: List[str]  # List of strings
    capacity: int

In [11]:
# Create a classroom
classroom = Classroom(
    room_number="A101",
    students=("Alice", "Bob", "Charlie"),
    capacity=30
)
print(classroom)

room_number='A101' students=['Alice', 'Bob', 'Charlie'] capacity=30


In [12]:
try:
    invalid_val=Classroom(room_number="A1",students=["Krish",123],capacity=30)
except ValueError as e:
    print(e)

1 validation error for Classroom
students.1
  Input should be a valid string [type=string_type, input_value=123, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type


#### 4. Model with Nested Models
Create complex structures with nested models:



In [13]:
from pydantic import BaseModel

class Address(BaseModel):
    street: str
    city: str
    zip_code: int

class Customer(BaseModel):
    customer_id: int
    name: str
    address: Address  # Nested model

# Create a customer with nested address
customer = Customer(
    customer_id=1,
    name="Emma",
    address={"street": "123 Main St", "city": "Boston", "zip_code": "02108"}
)
print(customer)

customer_id=1 name='Emma' address=Address(street='123 Main St', city='Boston', zip_code=2108)


#### Pydantic Fields: Customization and Constraints

The Field function in Pydantic enhances model fields beyond basic type hints by allowing you to specify validation rules, default values, aliases, and more. Here's a comprehensive tutorial with examples.





In [ ]:
from pydantic import BaseModel,Field
class Item(BaseModel):
    name:str=Field(min_length=2,max_length=50)
    price:float= Field(gt=0,le=1000) #greater than 0, less than or equal to 1000
    quantity:int=Field(ge=0)

# Valid instance
item = Item(name="Book", price=10, quantity=10)

print(item)


name='Book' price=10.0 quantity=10


In [ ]:
from pydantic import BaseModel, Field

class User(BaseModel):
    username: str = Field(..., description="Unique username for the user")
    age: int = Field(default=18, description="User age, defaults to 18")
    email: str = Field(default_factory=lambda: "user@example.com", description="Default email address")

# Examples
user1 = User(username="alice")
print(user1)  # username='alice' age=18 email='user@example.com'

user2 = User(username="bob", age=25, email="bob@domain.com")
print(user2)  # username='bob' age=25 email='bob@domain.com'

username='alice' age=18 email='user@example.com'
username='bob' age=25 email='bob@domain.com'


In [ ]:
print(User.model_json_schema())

{'properties': {'username': {'description': 'Unique username for the user', 'title': 'Username', 'type': 'string'}, 'age': {'default': 18, 'description': 'User age, defaults to 18', 'title': 'Age', 'type': 'integer'}, 'email': {'description': 'Default email address', 'title': 'Email', 'type': 'string'}}, 'required': ['username'], 'title': 'User', 'type': 'object'}
